In [3]:

# LAB 05 - MRI CNN ON PYNQ-Z2
# ARM only vs ARM + FPGA
# FPGA: Conv1 + ReLU + MaxPool1
# ARM : Remaining CNN layers

import time
import numpy as np
from PIL import Image
from pynq import Overlay, allocate
from tflite_runtime.interpreter import Interpreter

IMAGE = "brisc2025_test_00023_gl_ax_t1.jpg"
RUNS = 30
WARMUP = 5

# 1. Prepare MRI image (INT8, 96x96)
img = Image.open(IMAGE).convert("L")
img = img.resize((96, 96), Image.BILINEAR)
x = (np.asarray(img, dtype=np.int16) - 128)
x = x.astype(np.int8).reshape(1, 96, 96, 1)

# 2. Load FPGA and TFLite models
ol = Overlay("conv1_accel.bit")
dma = ol.axi_dma_0
conv1 = ol.conv1_hls_0

full = Interpreter(
    model_path="tiny_cnn_v5_int8.tflite",
    num_threads=1
)
tail = Interpreter(
    model_path="tiny_cnn_v5_tail_int8.tflite",
    num_threads=1
)

full.allocate_tensors()
tail.allocate_tensors()

fi = full.get_input_details()[0]["index"]
fo = full.get_output_details()[0]["index"]
ti = tail.get_input_details()[0]["index"]
to = tail.get_output_details()[0]["index"]

ibuf = allocate(shape=(9216,), dtype=np.int8)
obuf = allocate(shape=(47, 47, 8), dtype=np.int8)

# 3. Entire CNN on ARM
def run_arm():
    t0 = time.perf_counter()

    full.set_tensor(fi, x)
    full.invoke()
    result = full.get_tensor(fo)

    ms = (time.perf_counter() - t0) * 1000
    return result, ms

# 4. FPGA Conv1 + ARM remaining CNN
def run_hybrid():
    t0 = time.perf_counter()

    ibuf[:] = x.reshape(-1)
    ibuf.flush()

    dma.recvchannel.transfer(obuf)
    conv1.write(0x00, 1)
    dma.sendchannel.transfer(ibuf)

    # Poll DMA without artificial sleep
    deadline = time.monotonic() + 3.0

    while True:
        tx = dma.mmio.read(0x04)
        rx = dma.mmio.read(0x34)

        if (tx | rx) & 0x70:
            raise RuntimeError(
                f"DMA error: TX={hex(tx)}, RX={hex(rx)}"
            )

        if (tx & 2) and (rx & 2):
            break

        if time.monotonic() >= deadline:
            raise TimeoutError("DMA timeout")

    obuf.invalidate()
    t_fpga = time.perf_counter()

    pool1 = obuf.reshape(1, 47, 47, 8)
    tail.set_tensor(ti, pool1)
    tail.invoke()
    result = tail.get_tensor(to)

    t_end = time.perf_counter()

    return (
        result,
        (t_fpga - t0) * 1000,
        (t_end - t_fpga) * 1000,
        (t_end - t0) * 1000
    )

try:
    # 5. Warm-up
    print("Warming up...")

    for _ in range(WARMUP):
        a, _ = run_arm()
        b, _, _, _ = run_hybrid()
        assert np.array_equal(a, b), "Output mismatch"

    # 6. Benchmark with alternating execution order
    arm_times = []
    fpga_times = []
    tail_times = []
    hybrid_times = []

    print(f"Benchmarking {RUNS} runs...")

    for i in range(RUNS):
        if i % 2 == 0:
            a, arm_ms = run_arm()
            b, fpga_ms, tail_ms, hybrid_ms = run_hybrid()
        else:
            b, fpga_ms, tail_ms, hybrid_ms = run_hybrid()
            a, arm_ms = run_arm()

        if not np.array_equal(a, b):
            raise RuntimeError(f"Mismatch at run {i+1}")

        arm_times.append(arm_ms)
        fpga_times.append(fpga_ms)
        tail_times.append(tail_ms)
        hybrid_times.append(hybrid_ms)

    # 7. Results (median)
    arm_ms = np.median(arm_times)
    fpga_ms = np.median(fpga_times)
    tail_ms = np.median(tail_times)
    hybrid_ms = np.median(hybrid_times)

    scale, zp = tail.get_output_details()[0]["quantization"]
    scores = (b[0].astype(np.float32) - zp) * scale

    print("\n=== MRI CLASSIFICATION ===")
    print("Image:", IMAGE)
    print("Predicted class index:", int(np.argmax(scores)))
    print("Scores:", np.round(scores, 4))
    print("ARM output:", a.reshape(-1))
    print("Hybrid output:", b.reshape(-1))
    print("All outputs match: True")

    print("\n=== PERFORMANCE (MEDIAN) ===")
    print("Measured runs:", RUNS)
    print(f"ARM only       : {arm_ms:.3f} ms")
    print(f"FPGA + DMA     : {fpga_ms:.3f} ms")
    print(f"ARM tail       : {tail_ms:.3f} ms")
    print(f"ARM + FPGA     : {hybrid_ms:.3f} ms")
    print(f"Speedup        : {arm_ms/hybrid_ms:.3f}x")

    print("\nPASS: Hybrid CNN verified")

finally:
    ibuf.freebuffer()
    obuf.freebuffer()


Warming up...
Benchmarking 30 runs...

=== MRI CLASSIFICATION ===
Image: brisc2025_test_00023_gl_ax_t1.jpg
Predicted class index: 0
Scores: [0.9961 0.     0.     0.    ]
ARM output: [ 127 -128 -128 -128]
Hybrid output: [ 127 -128 -128 -128]
All outputs match: True

=== PERFORMANCE (MEDIAN) ===
Measured runs: 30
ARM only       : 23.626 ms
FPGA + DMA     : 1.507 ms
ARM tail       : 16.554 ms
ARM + FPGA     : 18.055 ms
Speedup        : 1.309x

PASS: Hybrid CNN verified
